# CAMB Wrapper: From Primordial Power Spectrum to CMB Angular Power Spectrum

This notebook explains how the CAMB wrapper (`scripts/camb_wrapper.py`) connects the primordial scalar power spectrum $P_\mathcal{R}(k)$ — computed by the inflation pipeline — to the CMB temperature angular power spectrum $C_\ell^{TT}$ that we compare against Planck 2018 data.

The pipeline is:

$$
\text{Background solver} \;\longrightarrow\; \text{Mukhanov-Sasaki} \;\longrightarrow\; P_\mathcal{R}(k) \;\xrightarrow{\text{CAMB}}\; C_\ell^{TT} \;\longrightarrow\; \chi^2 \;\text{vs Planck}
$$

## 1. Physical Background

### 1.1 The Angular Power Spectrum

The CMB temperature anisotropy $\Delta T(\hat{n})/\bar{T}$ is expanded in spherical harmonics:

$$
\frac{\Delta T(\hat{n})}{\bar{T}} = \sum_{\ell=2}^\infty \sum_{m=-\ell}^{\ell} a_{\ell m} Y_{\ell m}(\hat{n})
\tag{1}
$$

The angular power spectrum $C_\ell^{TT}$ is the variance of the multipole coefficients:

$$
C_\ell^{TT} = \langle |a_{\ell m}|^2 \rangle
\tag{2}
$$

For modern CMB analysis, the quantity $D_\ell = \ell(\ell+1)C_\ell/(2\pi) \times (T_\mathrm{cmb})^2$ is plotted in $\mu$K$^2$ units, where $T_\mathrm{cmb} = 2.7255$ K.

### 1.2 From Primordial Curvature to CMB Temperature

The CMB temperature today is a projection of the primordial curvature perturbation $\mathcal{R}(k)$ through a transfer function $\Delta_\ell(k)$:

$$
C_\ell^{TT} = 4\pi \int \frac{dk}{k} \, P_\mathcal{R}(k) \, |\Delta_\ell(k)|^2
\tag{3}
$$

where $P_\mathcal{R}(k) = \frac{k^3}{2\pi^2} \langle \mathcal{R}(k) \mathcal{R}(k') \rangle$ is the dimensionless primordial power spectrum from inflation.

### 1.3 The Transfer Function

CAMB (Code for Anisotropies in the Microwave Background) computes $\Delta_\ell(k)$ by solving the full Boltzmann + Einstein system, including:

- **Early Sachs-Wolfe effect**: gravitational redshift at last scattering
- **Acoustic oscillations**: photon-baryon fluid dynamics before recombination
- **Silk damping**: photon diffusion damping on small scales
- **Late Integrated Sachs-Wolfe (ISW) effect**: gravitational redshift from dark energy
- **CMB lensing**: gravitational lensing of CMB photons

The Sachs-Wolfe approximation ($\ell \lesssim 30$) is:

$$
C_\ell^{\mathrm{SW}} = \frac{4\pi}{25} \int \frac{dk}{k} \, P_\mathcal{R}(k) \, j_\ell^2(k r_\mathrm{ls})
\tag{4}
$$

where $r_\mathrm{ls} = 14000$ Mpc is the comoving distance to last scattering. The CAMB wrapper compares full and SW-only results via `run_comparison()` to isolate the ISW contribution.

### 1.4 Planck 2018 Comparison

For $\ell = 2$–$29$, we use the Commander likelihood (Planck 2018) with asymmetric 68% confidence limits. The goodness-of-fit is quantified by:

$$
\chi^2 = \sum_{i=2}^{29} \frac{(D_\ell^{\mathrm{model}} - D_\ell^{\mathrm{Planck}})^2}{\sigma_i^2}
\tag{5}
$$

where $\sigma_i = \sigma_i^+$ if the residual is positive, $\sigma_i = \sigma_i^-$ otherwise.

## 2. Importing the Codebase

All imports come from the project's `scripts/` and `models/` modules — no reimplementation of physics.

In [ ]:
import sys, os, glob, json
import numpy as np
import matplotlib.pyplot as plt

from models.higgs import HiggsModel
from pspectrum_pipeline import run_pspectrum_pipeline, load_pspectrum
from scripts.camb_wrapper import (
    compute_cl_full_camb,
    compute_cl_camb_powerlaw,
    compute_chi2_camb,
    run_comparison,
    save_camb_results,
)
from scripts.sachs_wolfe import compute_cl_sw
from scripts.planck_data import get_planck_data_asymmetric, C_ell_to_d_ell
from scripts.constants import As, k_pivot_phys, T_cmb

print("All imports successful.")
print(f"CAMB pivot scale: k_pivot = {k_pivot_phys} Mpc^-1")

## 3. Running the Higgs USR Pipeline

We instantiate the Higgs inflation model ($\xi = 15000$, $\lambda = 0.13$) with a representative configuration and compute $P_\mathcal{R}(k)$ via the full pipeline (background ODE → Mukhanov-Sasaki → power spectrum).

In [ ]:
# Golden config parameters
PHI0 = 6.80
Y0 = -0.60
NSTAR = 62.8

ROOT = os.path.join(os.getcwd(), "..")  # project root from notebooks/
ps_dir = os.path.join(ROOT, "outputs/simulations/pspectra")
cell_dir = os.path.join(ROOT, "outputs/simulations/c_ell")

# Try to load cached P_S(k)
ps_files = sorted(glob.glob(os.path.join(ps_dir, "*x06.80*y0-0.600*")))
ps_files = [f for f in ps_files if not os.path.basename(f).startswith("_checkpoint")]

if ps_files:
    cache_path = ps_files[-1]
    print(f"Loading cached P_S(k) from: {cache_path}")
    result = load_pspectrum(cache_path)
    meta = result["metadata"]
    print(f"  N_total={meta['N_total']:.1f}, N_pivot={meta['N_pivot']:.1f}, modes={len(result['k_phys'])}")

    # Try to load cached CAMB C_ell
    bn = os.path.splitext(os.path.basename(cache_path))[0]
    dell_path = os.path.join(cell_dir, f"C_ell_camb_{bn}.json")
    if os.path.exists(dell_path):
        print(f"Loading cached C_ell from: {dell_path}")
        with open(dell_path) as f:
            cached = json.load(f)
        ells = np.array(cached["c_ell"]["ells"])
        C_ell_TT = np.array(cached["c_ell"]["C_ell_TT"])
        C_ell_TE = np.array(cached["c_ell"]["C_ell_TE"])
        C_ell_EE = np.array(cached["c_ell"]["C_ell_EE"])
        D_ell = C_ell_to_d_ell(ells, C_ell_TT)
        print(f"  C_ell loaded: ell=2..{ells[-1]}")
    else:
        print("No cached C_ell found. Computing via CAMB...")
        ells, C_ell_TT, C_ell_TE, C_ell_EE = compute_cl_full_camb(result, ell_max=2500)
        D_ell = C_ell_to_d_ell(ells, C_ell_TT)
        os.makedirs(cell_dir, exist_ok=True)
        save_camb_results(ells, C_ell_TT, C_ell_TE, C_ell_EE, result["metadata"], dell_path)
        print(f"  C_ell cached to: {dell_path}")
else:
    print("No cached P_S(k) found. Running full pipeline...")
    model = HiggsModel(lam=0.13, xi=15000.0)
    model.phi0 = PHI0
    model.y0 = Y0
    result = run_pspectrum_pipeline(
        model,
        phi0=PHI0,
        y0=Y0,
        N_star=NSTAR,
        k_min=1e-5,
        k_max=1.0,
        num_k=120,
        normalize_to_As=True,
        As=As,
        save_outputs=True,
    )
    print(f"  Status: {result['status']}")
    print(f"  P_S(k) computed: {len(result['k_phys'])} k-modes")

    print("Computing C_ell via CAMB...")
    ells, C_ell_TT, C_ell_TE, C_ell_EE = compute_cl_full_camb(result, ell_max=2500)
    D_ell = C_ell_to_d_ell(ells, C_ell_TT)

    bn = os.path.splitext(os.path.basename(result["output_file"]))[0]
    dell_path = os.path.join(cell_dir, f"C_ell_camb_{bn}.json")
    os.makedirs(cell_dir, exist_ok=True)
    save_camb_results(ells, C_ell_TT, C_ell_TE, C_ell_EE, result["metadata"], dell_path)
    print(f"  C_ell cached to: {dell_path}")

print(f"k range: [{result['k_phys'].min():.2e}, {result['k_phys'].max():.2e}] Mpc^-1")

### 3.1 Visualizing the Primordial Spectrum

Plot $P_\mathcal{R}(k)$ to see the characteristic suppression caused by the USR phase.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

k = result['k_phys']
PS = result['P_S']

ax.loglog(k, PS, '.-', color='#4477AA', linewidth=1.5, markersize=3, label='Higgs USR')
ax.axvline(k_pivot_phys, color='gray', linestyle='--', alpha=0.5, label=f'$k_\mathrm{{pivot}} = {k_pivot_phys}$')
ax.axhline(As, color='gray', linestyle=':', alpha=0.5, label=f'$A_s = {As:.1e}$')

ax.set_xlabel('$k$ [Mpc$^{-1}$]', fontsize=14)
ax.set_ylabel('$P_\\mathcal{R}(k)$', fontsize=14)
ax.set_title('Primordial Scalar Power Spectrum', fontsize=16)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Computing $C_\ell$ via CAMB

The CAMB wrapper injects the computed $P_\mathcal{R}(k)$ via `set_initial_power_table()`, then solves the full Boltzmann hierarchy to produce $C_\ell^{TT}$, $C_\ell^{TE}$, and $C_\ell^{EE}$.

In [ ]:
# C_ell was already loaded or computed in the cache-aware cell above.
# If you want to recompute with different CAMB parameters, uncomment:
# ells, C_ell_TT, C_ell_TE, C_ell_EE = compute_cl_full_camb(result, ell_max=2500)
# D_ell = C_ell_to_d_ell(ells, C_ell_TT)

print(f"C_ell available: ell = 2..{ells[-1]}")
print(f"D_ell @ ell=2: {D_ell[0]:.2f} muK^2")
print(f"D_ell @ ell=10: {D_ell[8]:.2f} muK^2")
print(f"D_ell @ ell=100: {D_ell[98]:.2f} muK^2")

### 4.1 LCDM Baseline Comparison

Compute the same $C_\ell$ for a power-law $P_\mathcal{R}(k) = A_s (k/k_\mathrm{pivot})^{n_s-1}$ with Planck 2018 best-fit parameters ($A_s = 2.1\times 10^{-9}$, $n_s = 0.965$).

In [ ]:
ells_pl, C_ell_pl_TT, C_ell_pl_TE, C_ell_pl_EE = compute_cl_camb_powerlaw(
    ell_max=2500, As=As, ns=0.965
)
D_ell_pl = C_ell_to_d_ell(ells_pl, C_ell_pl_TT)

print("LCDM baseline (power-law) computed.")

## 5. Visualizing the Angular Power Spectrum

Plot full $D_\ell^{TT}$ from $\ell = 2$ to 2500, comparing Higgs USR (with its characteristic suppression) to the LCDM power-law baseline.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

# Full range
ax1.plot(ells, D_ell, color='#4477AA', linewidth=1.2, label='Higgs USR ($\\phi_0=6.80$, $y_0=-0.60$, $N_*=62.8$, $D_2=642$)')
ax1.plot(ells_pl, D_ell_pl, color='#CC6677', linewidth=1.2, linestyle='--', label='LCDM ($A_s=2.1$e-9, $n_s=0.965$)')

ax1.set_ylabel('$D_\\ell^{TT}$ [$\\mu$K$^2$]', fontsize=14)
ax1.set_title('CMB Temperature Angular Power Spectrum', fontsize=16)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Residual
ax2.plot(ells, D_ell - D_ell_pl, color='#4477AA', linewidth=1.0)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xlabel('$\\ell$', fontsize=14)
ax2.set_ylabel('$\\Delta D_\\ell$ [$\\mu$K$^2$]', fontsize=14)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### 5.1 Low-$\ell$ Region with Planck 2018 Data

Zoom into the low multipole range ($\ell \leq 30$) where the USR suppression is most visible, and overlay Planck 2018 Commander data with asymmetric error bars.

In [ ]:
planck_ells, D_planck, D_err_lower, D_err_upper = get_planck_data_asymmetric()

fig, ax = plt.subplots(figsize=(7, 4.5))

# Planck data points
ax.errorbar(planck_ells, D_planck,
            yerr=[D_err_upper, D_err_lower],
            fmt='o', color='k', markersize=4, capsize=2,
            label='Planck 2018 Commander')

# Higgs USR model
ax.plot(ells[:30], D_ell[:30], '.-', color='#4477AA', linewidth=1.5,
        label='Higgs USR')

# LCDM baseline
ax.plot(ells_pl[:30], D_ell_pl[:30], '--', color='#CC6677', linewidth=1.5,
        label='LCDM ($n_s=0.965$)')

ax.set_xlabel('$\\ell$', fontsize=14)
ax.set_ylabel('$D_\\ell^{TT}$ [$\\mu$K$^2$]', fontsize=14)
ax.set_title('Low-$\\ell$ CMB: Higgs USR vs Planck 2018', fontsize=16)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 5.2 Sachs-Wolfe vs Full CAMB: ISW Contribution

Compare the SW-only $C_\ell$ (valid at $\ell \lesssim 30$) with the full CAMB result to see the late ISW enhancement from dark energy.

In [ ]:
comp = run_comparison(result, ell_max=30)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(comp['ells'], comp['D_ell_SW'], '.-', color='#4477AA', linewidth=1.5,
        label='Sachs-Wolfe only')
ax.plot(comp['ells'], comp['D_ell_full'], '.-', color='#CC6677', linewidth=1.5,
        label='Full CAMB')
ax.plot(comp['ells'], comp['D_ell_ISW'], '.-', color='#44AA44', linewidth=1.5,
        label='ISW contribution')

ax.set_xlabel('$\\ell$', fontsize=14)
ax.set_ylabel('$D_\\ell^{TT}$ [$\\mu$K$^2$]', fontsize=14)
ax.set_title('Sachs-Wolfe vs Full CAMB: ISW Contribution', fontsize=16)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 5.3 $\chi^2$ vs Planck

Compute the $\chi^2$ statistic for the Higgs USR model relative to the LCDM baseline using asymmetric Planck errors.

In [ ]:
chi2_model, chi2_lcdm, delta_chi2 = compute_chi2_camb(result, ell_max=29)

print(f"chi2 (Higgs USR) = {chi2_model:.2f}")
print(f"chi2 (LCDM)      = {chi2_lcdm:.2f}")
print(f"Delta chi2       = {delta_chi2:+.2f}")
print()
print(f"Degrees of freedom: 28 (ell = 2..29)")
print(f"chi2/dof (Higgs):   {chi2_model/28:.2f}")
print(f"chi2/dof (LCDM):    {chi2_lcdm/28:.2f}")

## 6. Summary

### What we've shown

1. The **Higgs USR model** produces a scale-dependent suppression in $P_\mathcal{R}(k)$ from a brief ultra-slow-roll phase.
2. **CAMB** translates this $P_\mathcal{R}(k)$ into full $C_\ell^{TT}$ via the Boltzmann transfer function, capturing acoustic peaks, Silk damping, ISW, and lensing.
3. At low $\ell$ ($\lesssim 30$), the suppression of power reduces $D_\ell$ relative to the LCDM baseline, which can improve the fit to Planck 2018 data — quantified by the $\chi^2$ improvement.
4. The **ISW contribution** (CAMB minus SW-only) is positive at $\ell \lesssim 10$ due to dark energy.

### Physics insights

| Effect | Impact on $C_\ell$ | Physical origin |
|---|---|---|
| $P_\mathcal{R}(k)$ suppression | Reduces $D_\ell$ at low $\ell$ | USR phase $\to \epsilon_H \to 0 \to$ freezes curvature |
| Acoustic oscillations | Peaks at $\ell \sim 200$, $500$, etc. | Sound horizon at recombination |
| Silk damping | Exponential suppression $\ell \gtrsim 1500$ | Photon diffusion before decoupling |
| Late ISW | Boost at $\ell \lesssim 10$ | Dark energy-driven potential decay |
| CMB lensing | Smoothing of peaks $\ell \gtrsim 1000$ | Large-scale structure deflection |

### Free parameters

The Higgs USR model has three tunable initial conditions that control the dip location, depth, and width:
- $\phi_0$: starting field value on the plateau
- $y_0$: initial velocity (negative = kinetic dominance, $\epsilon_H \approx 2.15$)
- $N_*$: e-folds before end of inflation where $k_\mathrm{pivot}=0.05$ Mpc$^{-1}$ exits the horizon

The $\chi^2$ comparison against Planck 2018 low-$\ell$ data constrains these parameters, with the deepest-dip configuration $\phi_0=6.80$, $y_0=-0.60$, $N_*=62.8$ yielding $D_2=642$ (56\% suppression at $k_{\rm dip}=5.9\times 10^{-4}$). The best-fit configuration ($\phi_0=6.60$, $y_0=-0.736$, $N_*=52.6$) yields $\chi^2 = 16.79$ (LCDM baseline: $\chi^2 = 22.5$, $\ell = 2$–$29$).